# 08 · Las 10 preguntas ciegas (día 24)

**Objetivo:** Ejecutar las 10 preguntas ciegas sin tocar código y analizar el delta frente al golden propio.

**Requisitos:** R10, R15 ([01](../docs/01_requisitos_y_contratos.md)) · **Guía:** [13 §8](../docs/13_skill_medicion_informe_presentacion.md) · [12 §11](../docs/12_skill_evaluadores.md)

**Entradas:** fichero de preguntas ciegas del día 24 · **Salidas:** `resultados/ciegas/`

**Independiente:** se ejecuta solo, sin ejecutar antes otros notebooks: lee `data/` y lo guardado en `resultados/`. Con `EJECUTAR = False` no llama a la API.

In [1]:
# Arranque (igual en todos los notebooks): localiza la raíz del repo y hace importable src/
import json
import sys, pathlib
RAIZ = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ / "src"))
from IPython.display import Markdown, display
from agente10k import config
from agente10k.evaluacion import cargar_golden, evaluar

In [2]:
# Parámetros
EJECUTAR = False  # True: llama al modelo (gasta API y pide clave). False: reutiliza lo guardado en resultados/
RUTA_CIEGAS = config.RAIZ / "ciegas.jsonl"  # el fichero de las 10 preguntas ciegas del día 24
ETIQUETA_CIEGAS = "ciegas"                  # resultados/ciegas/{predicciones,puntuaciones,resumen}.jsonl|json
if EJECUTAR:
    assert config.cargar_clave(), "Falta OPENROUTER_API_KEY: copia .env.example a .env y rellénala"

## 1. Cargar el fichero de ciegas

`cargar_golden` no exige nada (docs/12 §2): admite tanto el golden propio como un JSONL sin `respuesta_esperada` ni `cifra_esperada`. Las
ciegas del día 24 llegan así — sin referencia — y `evaluadores.sin_referencia()` las reconoce en cuanto se ejecuten (docs/13 §3).

Guía: [12 §11](../docs/12_skill_evaluadores.md)

In [3]:
if RUTA_CIEGAS.is_file():
    preguntas_ciegas = cargar_golden(RUTA_CIEGAS)
    conteo_familias = {}
    for p in preguntas_ciegas:
        clave = p.get("familia") or "(sin familia)"
        conteo_familias[clave] = conteo_familias.get(clave, 0) + 1
    display(Markdown(f"**{len(preguntas_ciegas)} preguntas ciegas cargadas** de `{RUTA_CIEGAS.name}` · familias: {conteo_familias}"))
else:
    preguntas_ciegas = []
    display(Markdown(
        f"⚠️ **Falta `{RUTA_CIEGAS.name}` en la raíz del repo.** Este notebook no inventa las 10 preguntas ciegas: "
        "en cuanto el fichero del día 24 se coloque ahí (mismo formato JSONL que `golden/golden_propio.jsonl`, "
        "pero sin `respuesta_esperada` ni `cifra_esperada` — docs/12 §11), esta celda las carga sola, sin tocar código."))

⚠️ **Falta `ciegas.jsonl` en la raíz del repo.** Este notebook no inventa las 10 preguntas ciegas: en cuanto el fichero del día 24 se coloque ahí (mismo formato JSONL que `golden/golden_propio.jsonl`, pero sin `respuesta_esperada` ni `cifra_esperada` — docs/12 §11), esta celda las carga sola, sin tocar código.

## 2. Ejecutar

`evaluar(RUTA_CIEGAS, etiqueta='ciegas')` usa `sistema='final'` por defecto — el sistema de entrega — y guarda
`resultados/ciegas/{predicciones,puntuaciones,resumen}.jsonl|json`. Sin referencia, las filas quedan como
`familia="sin_referencia"` y `acierto=None`: no se inventa una respuesta esperada (docs/13 §3).

Guía: [13 §8](../docs/13_skill_medicion_informe_presentacion.md)

In [4]:
RUTA_RESUMEN_CIEGAS = config.RESULTADOS / ETIQUETA_CIEGAS / "resumen.json"

if EJECUTAR and preguntas_ciegas:
    puntuaciones_ciegas = evaluar(RUTA_CIEGAS, etiqueta=ETIQUETA_CIEGAS)   # sistema='final' por defecto
    display(puntuaciones_ciegas)
    resumen_ciegas = json.loads(RUTA_RESUMEN_CIEGAS.read_text(encoding="utf-8")) if RUTA_RESUMEN_CIEGAS.is_file() else {}
elif RUTA_RESUMEN_CIEGAS.is_file():
    resumen_ciegas = json.loads(RUTA_RESUMEN_CIEGAS.read_text(encoding="utf-8"))
    display(Markdown(f"Se reutiliza una ejecución ya guardada de `{ETIQUETA_CIEGAS}` "
                     f"({resumen_ciegas.get('n')} preguntas, {resumen_ciegas.get('fecha')})."))
else:
    resumen_ciegas = {}
    if not preguntas_ciegas:
        display(Markdown("`EJECUTAR=False` y sin `ciegas.jsonl`: nada que ejecutar todavía."))
    else:
        display(Markdown("`EJECUTAR=False`: no se llama al modelo. El día 24, cambia a `EJECUTAR=True` "
                         "para lanzar `evaluar()` sobre las 10 preguntas."))

`EJECUTAR=False` y sin `ciegas.jsonl`: nada que ejecutar todavía.

## 3. Delta frente al golden propio

Las ciegas no traen respuesta esperada, así que no hay un micro/macro directamente comparable con el golden propio. Lo que sí se puede
comparar sin inventar nada:

- **Métricas operativas** (latencia media, tokens medios, llamadas medias) del sistema de referencia (golden propio, 20 preguntas) frente a
  las ciegas: si el agente se comporta de forma muy distinta fuera del golden, es una señal en sí misma.
- **Lo verificable de las ciegas** (`resumen["sin_referencia"]`, docs/13 §3): tasa de verificación global, cifra frente a XBRL, cita vista y
  abstenciones, frente al `micro` del golden. No son la misma métrica, pero una tasa de verificación mucho más baja en las ciegas que el
  micro del golden es indicio de que parte de la mejora medida es memoria de esas 20 preguntas, no generalización.

Guía: [13 §8](../docs/13_skill_medicion_informe_presentacion.md) · [12 §11](../docs/12_skill_evaluadores.md)

In [5]:
import matplotlib.pyplot as plt

# Sistema de referencia del golden propio: 'final' en cuanto exista, si no el mejor disponible.
SISTEMA_REFERENCIA = next((s for s in ("final", "candidato_07", "baseline")
                           if (config.RESULTADOS / s / "resumen.json").is_file()), None)
resumen_referencia = (json.loads((config.RESULTADOS / SISTEMA_REFERENCIA / "resumen.json").read_text(encoding="utf-8"))
                      if SISTEMA_REFERENCIA else {})

if not resumen_ciegas:
    display(Markdown(
        f"**Pendiente:** sin `resultados/{ETIQUETA_CIEGAS}/resumen.json` no hay nada que comparar todavía. En "
        f"cuanto exista, esta celda compara sus métricas operativas contra `{SISTEMA_REFERENCIA or '(ningún sistema medido aún)'}` "
        "(golden propio, 20 preguntas) y su tasa de verificación (`sin_referencia`) frente al micro/macro del golden."))
elif not resumen_referencia:
    display(Markdown("**Pendiente:** todavía no hay ningún resumen de `baseline`, `candidato_07` o `final` con el que comparar."))
else:
    metricas = [("latencia_media_s", "Latencia media (s)"), ("tokens_medios", "Tokens medios"),
               ("llamadas_medias", "Llamadas medias")]
    fig, axes = plt.subplots(1, len(metricas), figsize=(12, 3.5))
    for ax, (clave, titulo) in zip(axes, metricas):
        valores = [resumen_referencia.get(clave), resumen_ciegas.get(clave)]
        etiquetas_x = [SISTEMA_REFERENCIA, ETIQUETA_CIEGAS]
        barras = ax.bar(etiquetas_x, [v if v is not None else 0 for v in valores], color=["#888888", "#8e44ad"])
        ax.bar_label(barras, labels=[f"{v:,.1f}" if v is not None else "n/d" for v in valores], padding=3)
        ax.set_title(titulo, fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

    sr = resumen_ciegas.get("sin_referencia") or {}
    display(Markdown(
        f"**Verificación de las ciegas (sin referencia, n={sr.get('n', '—')}):** verificación global "
        f"{sr.get('verificacion', '—')} · cifra frente a XBRL {sr.get('cifra_xbrl', '—')} · cita vista "
        f"{sr.get('cita_vista', '—')} · abstenciones {sr.get('abstenciones', '—')}.\n\n"
        f"**Micro del golden propio (`{SISTEMA_REFERENCIA}`):** {resumen_referencia.get('micro')}. No es la misma "
        "métrica — las ciegas no tienen referencia — pero sirve como lectura de qué parte de la mejora es "
        "generalización y qué parte es memoria del golden."))

**Pendiente:** sin `resultados/ciegas/resumen.json` no hay nada que comparar todavía. En cuanto exista, esta celda compara sus métricas operativas contra `candidato_07` (golden propio, 20 preguntas) y su tasa de verificación (`sin_referencia`) frente al micro/macro del golden.